# Ajuste de un modelo LLM BETO para una tarea de nivelación automática de textos de estudiantes de ELE.
NivELE en IberLEF 2026

## 1) Elegimos el conjunto de datos de opiniones de HuggingFace.


In [ ]:
from datasets import load_dataset_builder

In [ ]:
#Seleccionamos el dataset

ds_builder = load_dataset_builder("FJC/corpusELE.csv")

print("Descripción del dataset:", ds_builder.info.description)
print("Características (features) del dataset:", ds_builder.info.features)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Descripción del dataset: 
Características (features) del dataset: {'Unnamed: 0': Value('int64'), 'numero': Value('float64'), 'nivel': Value('string'), 'lenguaM': Value('string'), 'pClave': Value('string'), 'frase': Value('string'), 'archivo': Value('string')}


In [ ]:
from datasets import get_dataset_split_names

get_dataset_split_names("FJC/corpusELE.csv")


Repo card metadata block was not found. Setting CardData to empty.


['train']

El conjunto de datos tiene un solo subconjuntos: train.

## 2) Cargamos este conjunto con la librería de datasets.

In [ ]:
#Descargamos el dataset

from datasets import load_dataset

dataset = load_dataset("FJC/corpusELE.csv")
dataset

Repo card metadata block was not found. Setting CardData to empty.


corpusELE.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/46787 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'numero', 'nivel', 'lenguaM', 'pClave', 'frase', 'archivo'],
        num_rows: 46787
    })
})

In [ ]:
#Observamos las categorías:

LABELS = dataset['train'].unique('nivel')
NUM_LABELS = len(LABELS)
print('LABELS:', LABELS, 'num_labels:', NUM_LABELS )

LABELS: ['A1', 'B2', 'A2', 'B1', 'C1'] num_labels: 5


In [ ]:
#Contamos cuántos ejemplos hay de cada categoría

class_counts = dataset['train'].to_pandas()['nivel'].value_counts()
print(class_counts)

nivel
A2    13926
A1    12284
B1    10848
B2     6601
C1     3128
Name: count, dtype: int64


## 3) Fine-tuning de un modelo pre-entrenado.
### Ajustamos el modelo base para la tarea de clasificación de opiniones.

El modelo elegido ha sido BETO en español.


###  1.  Tokenización

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-cased")

In [ ]:
#Vemos cúal es el número máximo de tokens de nuestro dataset

MAX_LENGTH= max([len(tokenizer(text).input_ids) for text in dataset['train']['frase']])
print("Tamaño máximo", MAX_LENGTH)

Tamaño máximo 100


In [ ]:
#Tokenizamos por lotes:

def tokenize(examples):
    return tokenizer(examples["frase"], padding="max_length",max_length=MAX_LENGTH)

In [ ]:
#Aplicamos la función anterior a todos los splits:

encoded_data = dataset.map(tokenize, batched=True)
encoded_data

Map:   0%|          | 0/46787 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'numero', 'nivel', 'lenguaM', 'pClave', 'frase', 'archivo', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 46787
    })
})

In [ ]:
#Comprobamos que todos los textos tienen el mismo tamaño

import random
for i in range(10):
    index = random.randint(0,encoded_data['train'].num_rows)
    print('frase:', index, ' len:', len(encoded_data['train'][index]['input_ids']), 'nivel: ', encoded_data['train'][index]['nivel'])

KeyError: 'nivel'

In [ ]:
# Creamos los mapeos de etiquetas a IDs y viceversa
id2label = {idx: label for idx, label in enumerate(LABELS)}
label2id = {label: idx for idx, label in enumerate(LABELS)}

# Re-tokenize the dataset to ensure 'nivel' and 'frase' are present
encoded_data = dataset.map(tokenize, batched=True)

# Función para mapear las etiquetas y renombrar la columna
def map_labels_and_rename(example):
    example['labels'] = label2id[example['nivel']]
    return example

# Aplicamos la función a todo el dataset
encoded_data = encoded_data.map(map_labels_and_rename, batched=False)

# Renombramos la columna 'frase' a 'text'
encoded_data = encoded_data.rename_column("frase", "text")

# Eliminamos la columna original 'nivel' si ya no es necesaria y otras columnas no utilizadas por el modelo
encoded_data = encoded_data.remove_columns(['nivel', 'Unnamed: 0', 'numero', 'lenguaM', 'pClave', 'archivo'])

# Verificamos las nuevas características del dataset
print(encoded_data['train'].features)
print(encoded_data['train'][0])

Map:   0%|          | 0/46787 [00:00<?, ? examples/s]

Map:   0%|          | 0/46787 [00:00<?, ? examples/s]

{'text': Value('string'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8')), 'labels': Value('int64')}
{'text': 'mi familia es solo 3 persona , me madre y me hermano y mi .', 'input_ids': [4, 1153, 2268, 1058, 1942, 1244, 2274, 1017, 1129, 2489, 1042, 1129, 3216, 1042, 1153, 1009, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0

### 2. Cargamos el modelo y ajustamos los parámetros y las métricas de evaluación.

In [ ]:
#Cargamos el modelo:

from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("dccuchile/bert-base-spanish-wwm-cased", num_labels=NUM_LABELS)

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not 

In [ ]:
#Ajustamos los parámetros:

from transformers import TrainingArguments
args = TrainingArguments(output_dir="./outputs", label_names=['labels'])
args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,

In [ ]:
#Modificamos el tamaño del lote para entrenamiento y validación:

args.per_device_train_batch_size = 32
args.per_device_eval_batch_size = 32

In [ ]:
#Indicamos que la estrategia de entrenamiento para el modelo son epochs:

args.evaluation_strategy="epoch"
args.report_to='tensorboard'

In [ ]:
#Definimos las métricas para evaluar el modelo

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


def compute_metrics(pred):

    y_true = pred.label_ids                 # son las labels reales
    y_pred = pred.predictions.argmax(-1)    # son las predicciones


    acc = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


### 3. Creación y re-entrenamiento del modelo

In [ ]:
from transformers import Trainer

# Creamos el modelo

# Dividimos el conjunto de entrenamiento en entrenamiento y validación
train_validation_split = encoded_data['train'].train_test_split(test_size=0.1, seed=42)

trainer = Trainer(
    model = model,            # modelo que será ajustado
    train_dataset = train_validation_split['train'], # conjunto training
    eval_dataset = train_validation_split['test'],   # conjunto de validación

    args = args,     # hiperparámetros
    compute_metrics=compute_metrics    # función para computar las métricas
)

In [ ]:
print(f"Tamaño del conjunto de entrenamiento: {train_validation_split['train'].num_rows}")
print(f"Tamaño del conjunto de validación: {train_validation_split['test'].num_rows}")

Tamaño del conjunto de entrenamiento: 42108
Tamaño del conjunto de validación: 4679


In [ ]:
#y lo entrenamos

trainer.train()

Step,Training Loss
500,0.725814
1000,0.603096
1500,0.515561
2000,0.383324
2500,0.370537
3000,0.250109
3500,0.210168


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3948, training_loss=0.40959585473892535, metrics={'train_runtime': 2557.5433, 'train_samples_per_second': 49.393, 'train_steps_per_second': 1.544, 'total_flos': 6491823482152800.0, 'train_loss': 0.40959585473892535, 'epoch': 3.0})

### 4. Evaluación

In [ ]:
trainer.evaluate()

{'eval_loss': 0.6421806216239929,
 'eval_accuracy': 0.807651207522975,
 'eval_f1': 0.81082596787768,
 'eval_precision': 0.8132841826138314,
 'eval_recall': 0.8085965203167944,
 'eval_runtime': 32.3131,
 'eval_samples_per_second': 144.802,
 'eval_steps_per_second': 4.549,
 'epoch': 3.0}

### Identificación de ejemplos mal clasificados en el conjunto de validación

In [ ]:
# Obtener las predicciones del modelo para el conjunto de validación
predictions_output = trainer.predict(train_validation_split['test'])

# Las predicciones son los logits, necesitamos convertirlos a IDs de etiquetas
predicted_ids = predictions_output.predictions.argmax(axis=1)

# Obtener las etiquetas verdaderas del conjunto de validación
true_ids = predictions_output.label_ids

# Mapear los IDs de etiquetas a sus nombres originales (A1, A2, etc.)
predicted_labels_names = [id2label[id] for id in predicted_ids]
true_labels_names = [id2label[id] for id in true_ids]

# Crear un DataFrame para comparar fácilmente
comparison_df = pd.DataFrame({
    'text': train_validation_split['test']['text'],
    'true_label': true_labels_names,
    'predicted_label': predicted_labels_names
})

# Filtrar los ejemplos mal clasificados
misclassified_examples = comparison_df[comparison_df['true_label'] != comparison_df['predicted_label']]

print(f"Se encontraron {len(misclassified_examples)} ejemplos mal clasificados de un total de {len(train_validation_split['test'])} en el conjunto de validación.")

# Mostrar los primeros 10 ejemplos mal clasificados
print("\nPrimeros 10 ejemplos mal clasificados:")
display(misclassified_examples.head(10))

Se encontraron 900 ejemplos mal clasificados de un total de 4679 en el conjunto de validación.

Primeros 10 ejemplos mal clasificados:


,text,true_label,predicted_label
2,"El primero día , por la mañana , quitamos el e...",A2,B1
12,"Ella siempre sabes lo que quiere , es muy deci...",A1,A2
14,querida amiga,A2,A1
25,MI MUJER HA QUERIDO LA COMIDA Y TAMBIEN LA GEN...,A2,A1
31,Me agrada más las grandes ciudads e toda su mo...,A2,A1
36,Pero esto no faz tenemos siempre algo a encina...,A2,B2
39,"Mire , espero que le caiga mal .",B2,B1
47,"Entre otras cosas tambien dice , ¨ Yo no soy p...",C1,B1
48,"Jalid es de Arabia_Saudita , se trasladó a Din...",C1,A2
50,"De eso , ellos estan reflejando un cambio en e...",B2,C1


## 6) Carga del conjunto de datos de test y predicción.

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Ruta al archivo test.csv en Google Drive
test_file_path = '/content/drive/MyDrive/ColabNotebooks/APLICACIONES/ALBERTO/test.csv'

# Cargar el dataset de prueba
try:
    df_test = pd.read_csv(test_file_path)
    print("Dataset de test cargado correctamente:")
    display(df_test.head())
except FileNotFoundError:
    print(f"Error: El archivo '{test_file_path}' no fue encontrado. Asegúrate de que la ruta sea correcta y que Drive esté montado.")
    df_test = pd.DataFrame() # Crear un DataFrame vacío para evitar errores posteriores


Dataset de test cargado correctamente:


,id,text
0,0,Un bien pelicula que me gusta es mama mia. Es ...
1,1,yo estaba en barcelona es un ciudad buena fui ...
2,2,"Fiesta, feliz compleaños"
3,3,quiero participa en una fiesta
4,4,Mi pelicula prefiera en este momento es una an...


In [ ]:
# Asegurarse de que el DataFrame no esté vacío antes de procesar
if not df_test.empty:
    # Preparar el DataFrame de prueba para la predicción
    # Asumimos que la columna de texto se llama 'frase' como en el dataset original
    if 'frase' in df_test.columns:
        df_test = df_test.rename(columns={'frase': 'text'})
    elif 'text' not in df_test.columns:
        print("Advertencia: La columna 'frase' o 'text' no se encontró. Asegúrate de que el CSV contenga una columna con el texto a clasificar.")
        # Intentar usar la primera columna string disponible como 'text'
        string_cols = df_test.select_dtypes(include='object').columns
        if len(string_cols) > 0:
            df_test = df_test.rename(columns={string_cols[0]: 'text'})
            print(f"Se usará la columna '{string_cols[0]}' como 'text'.")
        else:
            print("Error: No hay columnas de texto en el dataset para hacer predicciones.")
            df_test = pd.DataFrame() # Vaciar el DataFrame si no hay texto

    # Asegurarse de tener una columna 'id' para el output
    if 'id' not in df_test.columns and not df_test.empty:
        df_test['id'] = df_test.index # Usar el índice como id si no hay una columna de id
        print("Se añadió una columna 'id' basada en el índice del DataFrame.")

    # Realizar predicciones si el DataFrame no está vacío y tiene la columna 'text'
    if not df_test.empty and 'text' in df_test.columns:
        print("Realizando predicciones en el dataset...")

        # Define the get_prediction function here to ensure it's available
        def get_prediction(text):
            # tokenizamos: prepara el texto, aplicamos la misma tokenización que la utilizada en el training; además añadimos truncation
            inputs = tokenizer(text, padding="max_length", max_length=MAX_LENGTH, truncation=True, return_tensors="pt").to(model.device)

            # aplicamos el modelo
            outputs = model(**inputs)

            # Pasamos a probabilidades usando softmax y obtenemos la clase con mayor probabilidad
            probs = outputs.logits.softmax(1)
            return probs.argmax().item()

        predicted_ids = [get_prediction(text) for text in df_test['text']]

        # Mapear los IDs predichos a sus etiquetas originales (A1, A2, B1, etc.)
        # 'id2label' se definió previamente en el notebook
        predicted_labels = [id2label[pred_id] for pred_id in predicted_ids]

        # Crear un DataFrame con las IDs y las etiquetas predichas
        results_df = pd.DataFrame({'id': df_test['id'], 'label': predicted_labels})

        print("Predicciones realizadas:")
        display(results_df.head())

        # Guardar los resultados en un nuevo archivo CSV en Google Drive
        output_file_path = '/content/drive/MyDrive/ColabNotebooks/APLICACIONES/ALBERTO/predictions2.csv'
        results_df.to_csv(output_file_path, index=False)
        print(f"Predicciones guardadas en '{output_file_path}'")
    else:
        print("No se pudieron realizar predicciones debido a un DataFrame de prueba vacío o sin columna 'text'.")
else:
    print("El DataFrame está vacío, no se realizarán más acciones.")


Realizando predicciones en el dataset de prueba...
Predicciones realizadas:


,id,label
0,0,A1
1,1,A2
2,2,A1
3,3,A1
4,4,C1


Predicciones guardadas en '/content/drive/MyDrive/ColabNotebooks/APLICACIONES/ALBERTO/predictions2.csv'
